# Experiment 4: Wine Clustering Model Practice

This notebook reuses the functions from `experiment4_wine_clustering.py`. The original `class` label is removed before clustering and is used only in the optional external comparison section.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

current_dir = Path.cwd().resolve()
experiment4_name = "\u5b9e\u9a8c\u56db"
if (current_dir / "experiment4_wine_clustering.py").exists():
    exp4_dir = current_dir
elif (current_dir / experiment4_name / "experiment4_wine_clustering.py").exists():
    exp4_dir = current_dir / experiment4_name
else:
    exp4_dir = Path(__file__).resolve().parent if "__file__" in globals() else current_dir

if str(exp4_dir) not in sys.path:
    sys.path.insert(0, str(exp4_dir))

from experiment4_wine_clustering import (
    resolve_paths,
    ensure_output_dirs,
    prepare_data,
    run_kmeans_experiment,
    run_agglomerative_experiment,
    run_dbscan_experiment,
    run_final_analysis,
)

paths = resolve_paths(exp4_dir)
ensure_output_dirs(paths)
paths

## Read data

Read `wine.data`, apply the required column names, and keep the original labels only as `y_true_optional`.

In [ ]:
data_outputs = prepare_data(paths)
data = data_outputs["data"]

df = data["df"]
X = data["X"]
y_true_optional = data["y_true_optional"]

df.head()

## Remove label

The first column `class` is removed before clustering. `X` contains only the 13 chemical features.

In [ ]:
print(f"class in X columns: {'class' in X.columns}")
print(f"X shape: {X.shape}")
print(f"y_true_optional shape: {y_true_optional.shape}")
X.head()

## Data checking

The script saves basic information, missing-value checks, duplicate-row checks, dtypes, and descriptive statistics.

In [ ]:
display(pd.read_csv(paths["results_dir"] / "data_basic_info.csv").head(20))
display(pd.read_csv(paths["results_dir"] / "missing_duplicate_check.csv").head(20))
display(pd.read_csv(paths["results_dir"] / "feature_descriptive_statistics.csv").head())

## Standardization

StandardScaler is applied to the feature matrix after the `class` label has been removed.

In [ ]:
X_scaled = data["X_scaled"]
X_scaled.describe().T.head()

## PCA visualization

PCA with two components is used only for visualization. The optional original-label PCA plot is not used for training, parameter selection, or main evaluation.

In [ ]:
X_pca_2d = data["X_pca_2d"]
pca = data["pca"]
print("Explained variance ratio:", pca.explained_variance_ratio_)
X_pca_2d.head()

## KMeans experiment

KMeans is tested for k from 2 to 10. The main metrics are Silhouette, Calinski-Harabasz, and Davies-Bouldin. Inertia is recorded only for KMeans k selection.

In [ ]:
kmeans_outputs = run_kmeans_experiment(data, paths)
kmeans_outputs["results"]

## Agglomerative clustering experiment

AgglomerativeClustering is tested with ward, complete, average, and single linkage for k from 2 to 10.

In [ ]:
agglomerative_outputs = run_agglomerative_experiment(data, paths)
agglomerative_outputs["results"].head()

## DBSCAN experiment

DBSCAN is tested over eps and min_samples. If there are fewer than two non-noise clusters, the internal metrics are recorded as NaN.

In [ ]:
dbscan_outputs = run_dbscan_experiment(data, paths)
dbscan_outputs["results"].head()

## Model comparison

Final comparison includes KMeans k=3, best KMeans by silhouette, Agglomerative ward k=3, best Agglomerative by silhouette, and the best valid DBSCAN setting.

In [ ]:
final_outputs = run_final_analysis(
    data,
    kmeans_outputs,
    agglomerative_outputs,
    dbscan_outputs,
    paths,
)
final_outputs["final_model_comparison"]

## Cluster profile analysis

KMeans k=3 is used for cluster profile interpretation in the original feature space and standardized feature space.

In [ ]:
display(final_outputs["cluster_profile_summary"])
display(final_outputs["cluster_profile_scaled_mean"])
print(final_outputs["cluster_profile_interpretation"])

## Optional label comparison

The original labels are used only after clustering as an optional external comparison. They are not used for training, parameter selection, or main evaluation.

In [ ]:
optional_external = pd.read_csv(paths["results_dir"] / "optional_external_label_comparison.csv")
optional_external

## Final conclusion

Use the generated `final_model_comparison.csv` and `model_ranking_summary.csv` for the final report. The recommendation below is based only on internal clustering metrics.

In [ ]:
print(final_outputs["analysis_text"])
print("
Generated results directory:", paths["results_dir"])
print("Generated figures directory:", paths["figures_dir"])